In [ ]:
import os
from getpass import getpass

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: "))
project = rf.workspace("hoainam-pusiu").project("nhan-dien-rac-dien-tu-ehsyl")
version = project.version(7)
dataset = version.download("yolo26")

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo26s.pt")

In [ ]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    
    # Training
    epochs=120,
    imgsz=640,
    batch=64,
    patience=30,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,              # learning rate cuối
    weight_decay=0.0005,

    # System
    device=[0,1],
    workers=4,
    cache=True,

    seed=42,
    pretrained=True,

    # LR schedule
    cos_lr=True,

    # Mosaic
    mosaic=1.0,
    close_mosaic=15,

    # Augmentation (QUAN TRỌNG)
    hsv_h=0.015,
    hsv_s=0.6,             # giảm nhẹ (0.7 hơi mạnh)
    hsv_v=0.4,

    degrees=7.0,           # giảm (10 hơi lệch nhiều)
    translate=0.08,        # giảm để không lệch object
    scale=0.4,             # giảm (0.5 hơi mạnh)
    
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,            # không lật dọc (không thực tế)

    mixup=0.1,             # thêm nhẹ (giúp generalize)

    # Output
    project="e_waste_detection",
    name="yolo26_optimized"
)

In [ ]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    
    # Training
    epochs=120,
    imgsz=640,
    batch=64,
    patience=30,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,              # learning rate cuối
    weight_decay=0.0005,

    # System
    device=[0,1],
    workers=4,
    cache=True,

    seed=42,
    pretrained=True,

    # LR schedule
    cos_lr=True,

    # Mosaic
    mosaic=1.0,
    close_mosaic=15,

    # Augmentation (QUAN TRỌNG)
    hsv_h=0.015,
    hsv_s=0.6,             # giảm nhẹ (0.7 hơi mạnh)
    hsv_v=0.4,

    degrees=7.0,           # giảm (10 hơi lệch nhiều)
    translate=0.08,        # giảm để không lệch object
    scale=0.4,             # giảm (0.5 hơi mạnh)
    
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,            # không lật dọc (không thực tế)

    mixup=0.1,             # thêm nhẹ (giúp generalize)

    # Output
    project="e_waste_detection",
    name="yolo26_optimized"
)

In [ ]:
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"   # QUAN TRỌNG: dùng test
)

print(metrics)

In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/runs/detect/e_waste_detection/yolo26_optimized2/weights/best.pt')

In [ ]:
import shutil

src = "/kaggle/working/runs/detect/e_waste_detection/yolo26_optimized/weights/best.pt"
dst = "/kaggle/working/best.pt"

shutil.copy(src, dst)

print("Đã copy xong, vào Output để tải best.pt")

In [ ]:
import cv2
from ultralytics import YOLO

# Load model
model = YOLO("train_yolo/best.pt")  # đường dẫn model của anh

# Mở webcam
cap = cv2.VideoCapture(0)

# (Tùy chọn) đặt độ phân giải
cap.set(3, 640)
cap.set(4, 480)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Predict
    results = model(frame, device=0)  # GPU (nếu có)

    # Vẽ kết quả
    annotated_frame = results[0].plot()

    # Hiển thị
    cv2.imshow("YOLO Webcam", annotated_frame)

    # Nhấn ESC để thoát
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO

# Load best model
best_model_path = f"{results.save_dir}/weights/best.pt"
model = YOLO(best_model_path)

# Evaluate trên test
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated())

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
model.predict(
    source=f"{dataset.location}/test/images",
    save=True,
    conf=0.25
)

In [ ]:
!zip -r yolo_train_results.zip runs/detect/e_waste_detection

In [ ]:
from IPython.display import FileLink
FileLink('yolo_train_results.zip')

In [ ]:
import os
import cv2
from tqdm import tqdm
dataset_path = dataset.location

print(dataset_path)

In [ ]:
splits = ["train", "valid", "test"]

cnn_dataset = "cnn_dataset"

In [ ]:
def crop_objects(image_path, label_path, save_dir):

    img = cv2.imread(image_path)
    h, w, _ = img.shape

    with open(label_path, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):

        parts = line.split()

        if len(parts) < 5:
            continue

        cls = float(parts[0])
        xc = float(parts[1])
        yc = float(parts[2])
        bw = float(parts[3])
        bh = float(parts[4])

        x_center = int(xc * w)
        y_center = int(yc * h)
        box_w = int(bw * w)
        box_h = int(bh * h)

        x1 = int(x_center - box_w / 2)
        y1 = int(y_center - box_h / 2)
        x2 = int(x_center + box_w / 2)
        y2 = int(y_center + box_h / 2)

        crop = img[y1:y2, x1:x2]

        class_dir = os.path.join(save_dir, str(int(cls)))
        os.makedirs(class_dir, exist_ok=True)

        filename = os.path.basename(image_path).replace(".jpg", f"_{i}.jpg")

        save_path = os.path.join(class_dir, filename)

        if crop.size > 0:
            cv2.imwrite(save_path, crop)

In [ ]:
for split in splits:

    image_dir = os.path.join(dataset_path, split, "images")
    label_dir = os.path.join(dataset_path, split, "labels")

    save_dir = os.path.join(cnn_dataset, split)

    images = os.listdir(image_dir)

    for img_name in tqdm(images):

        image_path = os.path.join(image_dir, img_name)

        label_path = os.path.join(
            label_dir,
            img_name.replace(".jpg", ".txt")
        )

        if os.path.exists(label_path):
            crop_objects(image_path, label_path, save_dir)

In [ ]:
import glob

for split in splits:

    count = len(glob.glob(f"cnn_dataset/{split}/*/*.jpg"))

    print(split, count)

# train cnn test 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [ ]:
dataset_path = "cnn_dataset"

train_dataset = datasets.ImageFolder(
    os.path.join(dataset_path,"train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(dataset_path,"valid"),
    transform=val_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(dataset_path,"test"),
    transform=val_transform
)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=4)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=4)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,num_workers=4)

In [ ]:
model = models.resnet50(pretrained=True)

num_classes = len(train_dataset.classes)

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0003)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def train_epoch(model, loader):

    model.train()
    total_loss = 0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, preds = torch.max(outputs,1)
        correct += torch.sum(preds==labels)

    acc = correct.double()/len(loader.dataset)

    return total_loss/len(loader), acc

In [ ]:
def evaluate(model, loader):

    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs,labels)

            total_loss += loss.item()

            _, preds = torch.max(outputs,1)

            correct += torch.sum(preds==labels)

    acc = correct.double()/len(loader.dataset)

    return total_loss/len(loader), acc

In [ ]:
best_val_acc = 0
epochs = 20

for epoch in range(epochs):

    train_loss, train_acc = train_epoch(model, train_loader)

    val_loss, val_acc = evaluate(model, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1}")
    print("Train Loss:",train_loss,"Train Acc:",train_acc)
    print("Val Loss:",val_loss,"Val Acc:",val_acc)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(model.state_dict(),"best_cnn_model.pth")

In [ ]:
model.load_state_dict(torch.load("best_cnn_model.pth"))

test_loss, test_acc = evaluate(model, test_loader)

print("Test Accuracy:",test_acc)